In [1]:
import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm
from sklearn.metrics import classification_report
import numpy as np
import accelerate

In [2]:
print("--- 1. Loading the Zero-Shot NLI Pipeline ---")
# We use a state-of-the-art multilingual NLI model that understands French natively
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=0,  # Sets it to use GPU/Apple Silicon if available, otherwise remove this line or set to -1 for CPU
)

--- 1. Loading the Zero-Shot NLI Pipeline ---


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
print("\n--- 2. Semantic Label Mapping ---")
# We MUST translate your technical targets into natural French phrases for the LLM
label_map = {
    "LEFT_PS": "le Parti Socialiste et la gauche",
    "LEFT_PCF": "le Parti Communiste",
    "FAR_LEFT": "l'extrême gauche et la lutte ouvrière",
    "ECO": "l'écologie et l'environnement",
    "RIGHT_RPR": "le Rassemblement pour la République et la droite gaulliste",
    "RIGHT_UDF": "l'Union pour la Démocratie Française et le centre droit",
    "RIGHT_UPF": "l'union de la droite",
    "RIGHT_URC": "le rassemblement du centre",
    "RIGHT_UNM": "la majorité présidentielle de droite",
    "FAR_RIGHT_FN": "le Front National et l'extrême droite",
}

# The candidate labels the model will actually see
candidate_labels = list(label_map.values())

# A reverse dictionary to map the model's textual prediction back to your technical labels for scoring
reverse_label_map = {v: k for k, v in label_map.items()}


--- 2. Semantic Label Mapping ---


In [4]:
print("\n--- 3. Loading Test Data ---")
# Load the hold-out test set (so we can compare directly against your SVM baseline)
df_test = pd.read_csv("../data/processed/X_test_raw.csv")
y_test = np.load("../data/processed/y_test.npy", allow_pickle=True)

# Zero-shot is computationally heavy. Let's prototype on just 50 random test documents first.
sample_indices = np.random.choice(len(df_test), 200, replace=False)
sample_texts = df_test["text"].iloc[sample_indices].fillna("").tolist()
sample_y_true = y_test[sample_indices]


--- 3. Loading Test Data ---


In [ ]:
print("\n--- 4. Running Zero-Shot Inference ---")
y_pred_zero_shot = []

for text in tqdm(sample_texts, desc="Zero-Shot Classification"):
    # We truncate the text to the first ~400 words to fit the transformer's context window
    truncated_text = " ".join(text.split()[:400])

    # The pipeline calculates probabilities for all candidate labels
    result = classifier(truncated_text, candidate_labels)

    # Get the top predicted natural language label
    top_label_text = result["labels"][0]

    # Map it back to the technical label (e.g., "LEFT_PS")
    technical_label = reverse_label_map[top_label_text]
    y_pred_zero_shot.append(technical_label)

print("\n--- 5. Zero-Shot Results ---")
print(classification_report(sample_y_true, y_pred_zero_shot, zero_division=0))


--- 4. Running Zero-Shot Inference ---


Zero-Shot Classification:   0%|          | 0/200 [00:00<?, ?it/s]


--- 5. Zero-Shot Prototype Results ---
              precision    recall  f1-score   support

         ECO       0.56      0.97      0.71        33
    FAR_LEFT       1.00      0.83      0.91        12
FAR_RIGHT_FN       0.00      0.00      0.00        39
    LEFT_PCF       0.75      0.31      0.44        39
     LEFT_PS       0.42      0.57      0.48        35
   RIGHT_RPR       0.00      0.00      0.00        10
   RIGHT_UDF       0.00      0.00      0.00         7
   RIGHT_UNM       0.00      0.00      0.00         2
   RIGHT_UPF       0.10      0.38      0.16         8
   RIGHT_URC       0.36      0.93      0.52        15

    accuracy                           0.46       200
   macro avg       0.32      0.40      0.32       200
weighted avg       0.40      0.46      0.39       200



In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset

print("--- 1. Hardware Check ---")
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Training on: {str(device).upper()}")

print("\n--- 2. Loading Exact SVM Splits (Processed Text) ---")
# Loading the exact splits saved during the feature engineering step
X_train_processed = (
    pd.read_csv("../data/processed/X_train_raw.csv")["text"].fillna("").tolist()
)
X_test_processed = (
    pd.read_csv("../data/processed/X_test_raw.csv")["text"].fillna("").tolist()
)

y_train_str = np.load("../data/processed/y_train.npy", allow_pickle=True)
y_test_str = np.load("../data/processed/y_test.npy", allow_pickle=True)

# Encode Labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_str)
y_test_encoded = le.transform(y_test_str)
num_labels = len(le.classes_)

# PROTOTYPE TOGGLE: Set to 500 for a quick test, or None for the full 6,400 dataset
prototype_size = 500
if prototype_size:
    print(f"Subsetting to {prototype_size} documents for prototyping...")
    X_train_processed = X_train_processed[:prototype_size]
    y_train_encoded = y_train_encoded[:prototype_size]
    X_test_processed = X_test_processed[:prototype_size]
    y_test_encoded = y_test_encoded[:prototype_size]

train_dataset = Dataset.from_dict({"text": X_train_processed, "label": y_train_encoded})
test_dataset = Dataset.from_dict({"text": X_test_processed, "label": y_test_encoded})

print("\n--- 3. Computing Class Weights ---")
# Calculate weights to heavily penalize missing the minority classes
weights = compute_class_weight(
    "balanced", classes=np.unique(y_train_encoded), y=y_train_encoded
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f"Class Weights Computed.")

print("\n--- 4. Tokenization & Model Initialization ---")
model_name = "almanach/camembert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=512
    )


print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label={i: label for i, label in enumerate(le.classes_)},
    label2id={label: i for i, label in enumerate(le.classes_)},
)

# LINEAR PROBING: Freeze the base model
for param in model.roberta.parameters():
    param.requires_grad = False

print("\n--- 5. Training Configuration (Aggressive LR & Early Stopping) ---")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1_macro": macro_f1}


# Subclass the Trainer to inject our custom class weights into the PyTorch Loss Function
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir="../models/camembert_linear_probe",
    eval_strategy="epoch",
    learning_rate=1e-3,  # Aggressive learning rate for random head
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,  # High ceiling, rely on Early Stopping
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,  # REQUIRED for Early Stopping
    metric_for_best_model="f1_macro",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("\n--- 6. Beginning Training Execution ---")
trainer.train()

print("\nTraining Complete! Best model restored.")

# Final Evaluation on Test Set
print("\n--- 7. Final Evaluation ---")
prediction_output = trainer.predict(tokenized_test)
metrics = prediction_output.metrics

print(f"Final Test Accuracy: {metrics['test_accuracy']:.4f}")
print(f"Final Test F1 Macro: {metrics['test_f1_macro']:.4f}")

--- 1. Hardware Check ---
Training on: MPS

--- 2. Loading Exact SVM Splits (Processed Text) ---
Subsetting to 500 documents for prototyping...

--- 3. Computing Class Weights ---
Class Weights Computed.

--- 4. Tokenization & Model Initialization ---
Tokenizing datasets...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: almanach/camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/Users/audricsicard/Documents/VSCode/ML for NLP/Project/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training Complete! Best model restored.

--- 7. Final Evaluation ---


RuntimeError: on_train_begin must be called before on_evaluate